In [ ]:
from pathlib import Path

from PIL import Image
from speciesnet import DEFAULT_MODEL, SpeciesNet
from ultralytics import YOLO



In [ ]:
class SpeciesNetYoloClassifier:
    def __init__(self, speciesnet_model, classifier_weights):
        # SpeciesNet's detector gives us MegaDetector-style normalized boxes.
        self.detector = SpeciesNet(speciesnet_model, components="detector")
        self.classify = YOLO(classifier_weights, task="classify")

    def _detect(self, image_path):
        ...

    def _classify(self, crop):
        ...

    def predict(self, image_path):
        image_path = Path(image_path).resolve()
        detection = self.detector.predict(filepaths=[str(image_path)], batch_size=1)
        detections = detection["predictions"][0].get("detections", [])
        if not detections:
            return {"probabilities": {}, "result": None, "confidence": None}

        # Keep the single highest-confidence detection.
        box = max(detections, key=lambda item: item["conf"])["bbox"]
        x, y, width, height = box
        with Image.open(image_path) as image:
            image = image.convert("RGB")
            crop = image.crop((
                x * image.width,
                y * image.height,
                (x + width) * image.width,
                (y + height) * image.height,
            ))

        classification = self.classify(crop, verbose=False)[0]
        probabilities = {
            classification.names[index]: float(probability)
            for index, probability in enumerate(classification.probs.data.cpu().tolist())
        }
        top1 = int(classification.probs.top1)
        return {
            "probabilities": probabilities,
            "result": classification.names[top1],
            "confidence": float(classification.probs.top1conf),
        }

In [ ]:
model = SpeciesNetYoloClassifier("kaggle:google/speciesnet/pyTorch/v4.0.3b/1", "some/path/yolo26cls.pt")
model.predict("/Users/hmack/Development/smartrodent_experiments/datasets/rgb/image_20260701T032219Z.jpg")